# Libraries

In [24]:
import pandas as pd
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [2]:
df = pd.read_csv("database\\model_data.csv")
df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'])

In [3]:
df

,FECHA_HORA,IDELEM,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,2017-01-01 00:00:00,3910,129.000000,1.000000,3.750000,40.0
1,2017-01-01 00:00:00,3911,151.000000,0.666667,3.666667,40.0
2,2017-01-01 00:00:00,3913,106.750000,0.000000,3.000000,40.0
3,2017-01-01 00:00:00,3914,145.500000,0.250000,3.750000,40.0
4,2017-01-01 00:00:00,3915,118.250000,0.250000,2.750000,40.0
...,...,...,...,...,...,...
464305,2024-12-31 23:00:00,3913,57.333333,0.000000,1.333333,62.0
464306,2024-12-31 23:00:00,3914,56.500000,0.000000,1.250000,62.0
464307,2024-12-31 23:00:00,3915,43.000000,0.000000,0.666667,62.0
464308,2024-12-31 23:00:00,3917,50.000000,0.000000,1.000000,62.0


In [4]:
# Me interesa FECHA_HORA para separar en entrenamiento y test
df['HORA'] = df['FECHA_HORA'].dt.hour

In [5]:
df

,FECHA_HORA,IDELEM,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION,HORA
0,2017-01-01 00:00:00,3910,129.000000,1.000000,3.750000,40.0,0
1,2017-01-01 00:00:00,3911,151.000000,0.666667,3.666667,40.0,0
2,2017-01-01 00:00:00,3913,106.750000,0.000000,3.000000,40.0,0
3,2017-01-01 00:00:00,3914,145.500000,0.250000,3.750000,40.0,0
4,2017-01-01 00:00:00,3915,118.250000,0.250000,2.750000,40.0,0
...,...,...,...,...,...,...,...
464305,2024-12-31 23:00:00,3913,57.333333,0.000000,1.333333,62.0,23
464306,2024-12-31 23:00:00,3914,56.500000,0.000000,1.250000,62.0,23
464307,2024-12-31 23:00:00,3915,43.000000,0.000000,0.666667,62.0,23
464308,2024-12-31 23:00:00,3917,50.000000,0.000000,1.000000,62.0,23


In [37]:
train = df[df['FECHA_HORA'] < '2024-01-01']
test = df[df['FECHA_HORA'] >= '2024-01-01']

In [38]:
train = train.groupby(['HORA']).agg({ # train.groupby(['IDELEM','HORA']).agg({
    'INTENSIDAD': 'mean',
    'OCUPACION': 'mean',
    'CARGA': 'mean',
    'VALOR_CONTAMINACION': 'mean'
}).reset_index()

In [39]:
train

,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,0,195.030295,1.168680,7.390287,36.373619
1,1,126.915316,0.740010,4.823568,29.383156
2,2,81.240826,0.434042,2.864462,23.118953
3,3,58.481206,0.331532,1.919874,19.400037
4,4,46.717968,0.306666,1.423832,17.099157
5,5,46.582796,0.268332,1.277559,17.249045
6,6,100.044143,0.489134,2.640945,23.998404
7,7,308.633810,1.723027,9.444120,38.345121
8,8,703.616703,5.863865,26.439154,47.355108
9,9,776.705052,7.049412,31.737022,43.908783


In [40]:
test = test.groupby(['HORA']).agg({ # test.groupby(['IDELEM','HORA']).agg({
    'INTENSIDAD': 'mean',
    'OCUPACION': 'mean',
    'CARGA': 'mean',
    'VALOR_CONTAMINACION': 'mean'
}).reset_index()

In [41]:
test

,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,0,208.321107,1.218308,7.928836,26.643639
1,1,129.887036,0.694062,4.978450,22.535076
2,2,79.656207,0.397119,2.783693,17.732510
3,3,55.694130,0.242284,1.777492,14.949931
4,4,45.044790,0.176152,1.290836,13.054677
5,5,46.735557,0.170134,1.265303,12.931224
6,6,96.355055,0.347432,2.469194,18.136176
7,7,273.028892,1.426012,8.385031,29.321331
8,8,681.223894,6.022458,25.611235,35.002394
9,9,766.348700,8.223201,31.699966,31.497092


In [42]:
features = ['HORA', 'INTENSIDAD', 'OCUPACION', 'CARGA'] # ['IDELEM', 'HORA', 'INTENSIDAD', 'OCUPACION', 'CARGA']
target = ['VALOR_CONTAMINACION']

In [43]:
X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

In [44]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

c:\Users\andre\Documents\Master\TFM\spark_anomaly_detection\.venv\Lib\site-packages\sklearn\base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [45]:
# Save model to pickle
import pickle
with open('database\\models\\pollution_model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [46]:
# Load model
with open('database\\models\\pollution_model.pkl', 'rb') as f:
    model = pickle.load(f)

In [47]:
y_pred = model.predict(X_test)

In [48]:
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

MSE: 102.00
RMSE: 10.10


In [49]:
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")

MAE: 9.51


In [50]:
r2 = r2_score(y_test, y_pred)
print(f"R-cuadrado (R²): {r2:.2f}")

R-cuadrado (R²): -0.78


# Repeat process but SCALING the data

In [51]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scale_columns = ['INTENSIDAD', 'OCUPACION', 'CARGA']
train[scale_columns] = scaler.fit_transform(train[scale_columns])
test[scale_columns] = scaler.fit_transform(test[scale_columns])

In [52]:
train

,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,0,-0.987329,-1.028003,-0.966323,36.373619
1,1,-1.229602,-1.211331,-1.191076,29.383156
2,2,-1.392058,-1.342183,-1.362624,23.118953
3,3,-1.473009,-1.386023,-1.445337,19.400037
4,4,-1.514849,-1.396658,-1.488772,17.099157
5,5,-1.515330,-1.413052,-1.501581,17.249045
6,6,-1.325178,-1.318622,-1.382196,23.998404
7,7,-0.583262,-0.790927,-0.786480,38.345121
8,8,0.821620,0.979972,0.701681,47.355108
9,9,1.081582,1.486991,1.165586,43.908783


In [53]:
test

,HORA,INTENSIDAD,OCUPACION,CARGA,VALOR_CONTAMINACION
0,0,-0.916881,-0.933957,-0.895632,26.643639
1,1,-1.200270,-1.152236,-1.156831,22.535076
2,2,-1.381758,-1.275874,-1.351134,17.732510
3,3,-1.468334,-1.340342,-1.440213,14.949931
4,4,-1.506811,-1.367878,-1.483297,13.054677
5,5,-1.500703,-1.370383,-1.485558,12.931224
6,6,-1.321423,-1.296562,-1.378977,18.136176
7,7,-0.683087,-0.847476,-0.855245,29.321331
8,8,0.791754,1.066335,0.669798,35.002394
9,9,1.099317,1.982653,1.208836,31.497092


In [54]:
X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

In [55]:
model_scaled = RandomForestRegressor()
model_scaled.fit(X_train, y_train)

c:\Users\andre\Documents\Master\TFM\spark_anomaly_detection\.venv\Lib\site-packages\sklearn\base.py:1363: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [56]:
with open('database\\models\\pollution_model_scaled.pkl', 'wb') as f:
    pickle.dump(model_scaled, f)

In [57]:
with open('database\\models\\pollution_model_scaled.pkl', 'rb') as f:
    model_scaled = pickle.load(f)

In [58]:
y_pred = model_scaled.predict(X_test)

In [59]:
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

MSE: 106.92
RMSE: 10.34


In [60]:
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")

MAE: 9.88


In [61]:
r2 = r2_score(y_test, y_pred)
print(f"R-cuadrado (R²): {r2:.2f}")

R-cuadrado (R²): -0.87


No se si el r2 sirve en regresion, creo que no